# Prompt Preview

Ce notebook lit `input/swissdox_2025_tagged.csv`, formate chaque article dans le template de prompt défini dans `src/run3_prompts.py` (avec mots-clés institutionnels), et produit un CSV avec :
- **`prompt_ready`** : le prompt complet à coller dans un chat LLM
- **`llm_answer`** : colonne vide à remplir manuellement avec la réponse du LLM

La colonne `text` brute est exclue pour alléger le fichier.

In [1]:
import sys
import pandas as pd
from pathlib import Path

# Ajouter src/ au path pour importer run3_prompts.py
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from run3_prompts import build_user_prompt, SYSTEM_PROMPT

In [2]:
# Charger les données
df = pd.read_csv('../input/swissdox_2025_tagged.csv')
print(f"Articles chargés : {len(df):,}")
print(f"Colonnes : {df.columns.tolist()}")
df.head(2)

Articles chargés : 23,665
Colonnes : ['article_id', 'title', 'lead', 'text', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'matched_keywords']


,article_id,title,lead,text,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,language,char_count,dateline,matched_keywords
0,0001de76-bb46-1fd0-ff60-980d765e15a8,Mega-Angriff legt Schweizer Seiten lahm: Hacke...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,2025-01-21,ZWAO,20 minuten online,NaN,NaN,WWE,Online medium,de,2666.0,NaN,Behörden
1,00027f05-690b-f341-5519-fcc50dbc5549,Martin Pfister envisage l’abandon du projet à ...,Le Conseiller fédéral envisage d’abandonner l’...,Le Conseiller fédéral envisage d’abandonner l’...,2025-07-05,ZWSO,20 minutes online,NaN,NaN,WWE,Online medium,fr,2813.0,NaN,Martin Pfister|Armée suisse


In [3]:
# Construire la colonne prompt_ready (system prompt + user prompt avec mots-clés)
df['prompt_ready'] = df.apply(
    lambda row: SYSTEM_PROMPT + "\n\n" + build_user_prompt(row, 'text'), axis=1
)

# Ajouter la colonne vide pour les réponses LLM
df['llm_answer'] = ''

print(f"Exemple de prompt_ready pour le premier article :")
print("-" * 60)
print(df['prompt_ready'].iloc[0])

Exemple de prompt_ready pour le premier article :
------------------------------------------------------------


You will receive an article to analyze. Your task is to tell if "" is being criticized in this article. Criticism can be express even if the overall evaluation is positive in the article. The criticism can be about who it is or what it did. Answer ONLY by YES or NO nothing else

ARTICLE:
NoName057(16) bekennt sich zu DDoS-Angriffen auf Schweizer Banken und Gemeinden. Die pro-russische Gruppe testet die Internet-Infrastruktur. Jonas Bucher Darum gehts Eine russische Hackergruppe hat erneut Schweizer Websites angegriffen. Betroffen sind die Zürcher und Waadtländer Kantonalbanken sowie mehrere Gemeinden. Die Gruppe NoName057(16) bekennt sich zu den DDoS-Angriffen auf der Plattform X. Die Angriffe sind politisch motiviert und richten sich gegen Länder, die die Ukraine unterstützen. Seit Monaten werden Websites von Schweizer Banken und Behörden Ziel von sogenannten DDoS-Angriffen

In [4]:
# Supprimer la colonne text brute et exporter
cols_to_keep = [c for c in df.columns if c != 'text']
df_out = df[cols_to_keep]

out_path = Path('../data/raw/swissdox_prompts_run3.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False)

print(f"Fichier exporté : {out_path}")
print(f"Colonnes : {df_out.columns.tolist()}")
print(f"Lignes : {len(df_out):,}")

Fichier exporté : ../data/raw/swissdox_prompts_run3.csv
Colonnes : ['article_id', 'title', 'lead', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'matched_keywords', 'prompt_ready', 'llm_answer']
Lignes : 23,665


In [5]:
# Aperçu du résultat final
df_out[['article_id', 'medium_name', 'pubtime', 'matched_keywords', 'prompt_ready', 'llm_answer']].head(3)

,article_id,medium_name,pubtime,matched_keywords,prompt_ready,llm_answer
0,0001de76-bb46-1fd0-ff60-980d765e15a8,20 minuten online,2025-01-21,Behörden,\n\nYou will receive an article to analyze. Yo...,
1,00027f05-690b-f341-5519-fcc50dbc5549,20 minutes online,2025-07-05,Martin Pfister|Armée suisse,\n\nYou will receive an article to analyze. Yo...,
2,000658f2-9b7b-c9b4-9592-008f8f4d0b54,24 heures,2025-09-30,Administration fédérale|Fonctionnaires|Guy Par...,\n\nYou will receive an article to analyze. Yo...,
